In [ ]:
# ── Ячейка 1: Установка ──────────────────────────────────────────────────────
!pip install openai-whisper -q

In [ ]:
# ── Ячейка 2: Импорты и конфиг ───────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import json
import pickle
import torch
import whisper
from pathlib import Path
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WHISPER_MODEL = "large-v3"  # large-v3 для максимального качества

DATA_DIR = Path("/kaggle/input/competitions/multi-lingual-video-fragment-retrieval-challenge/video-rag")
AUDIO_FILES_CSV = DATA_DIR / "audio_files.csv"
OUTPUT_FILE = Path("transcripts_large.pkl")  # не перезаписываем оригинальный transcripts.pkl
BATCH_SAVE = 10

print(f"Device: {DEVICE}")
print(f"Whisper model: {WHISPER_MODEL}")

In [ ]:
# ── Ячейка 3: Загрузка списка аудиофайлов ────────────────────────────────────
import pandas as pd

audio_df = pd.read_csv(AUDIO_FILES_CSV)
audio_paths = [DATA_DIR / p for p in audio_df["audio_path"].tolist()]
print(f"Всего аудиофайлов: {len(audio_paths)}")

In [ ]:
# ── Ячейка 4: Загрузка модели ────────────────────────────────────────────────
print(f"Загружаем Whisper {WHISPER_MODEL}...")
model = whisper.load_model(WHISPER_MODEL, device=DEVICE)
print("Готово")

In [ ]:
# ── Ячейка 5: Транскрипция с resume ──────────────────────────────────────────

# Resume: загружаем уже готовые если есть
if OUTPUT_FILE.exists():
    with open(OUTPUT_FILE, "rb") as f:
        transcripts = pickle.load(f)
    print(f"Уже обработано: {len(transcripts)}, осталось: {len(audio_paths) - len(transcripts)}")
else:
    transcripts = {}

for audio_path in tqdm(audio_paths, desc="Транскрипция"):
    if not audio_path.exists():
        print(f"  [ПРОПУСК] не найден: {audio_path}")
        continue

    # Ключ — путь к видео (как в оригинальном transcripts.pkl)
    audio_name = audio_path.name
    video_name = audio_name.replace("audio_", "video_")
    video_key = f"videos/{Path(video_name).stem}"  # "videos/video_abc123"

    if video_key in transcripts:
        continue  # уже обработан

    try:
        result = model.transcribe(
            str(audio_path),
            word_timestamps=False,
            verbose=None,
            beam_size=5,
        )

        segments = [
            {
                "start": seg["start"],
                "end":   seg["end"],
                "text":  seg["text"].strip(),
            }
            for seg in result["segments"]
            if seg["text"].strip()
        ]

        transcripts[video_key] = segments

    except Exception as e:
        print(f"  [ОШИБКА] {audio_path.name}: {e}")
        transcripts[video_key] = []

    # Сохраняем каждые 10 файлов
    if len(transcripts) % BATCH_SAVE == 0:
        with open(OUTPUT_FILE, "wb") as f:
            pickle.dump(transcripts, f)
        print(f"  [сохранено {len(transcripts)} файлов]")

# Финальное сохранение
with open(OUTPUT_FILE, "wb") as f:
    pickle.dump(transcripts, f)

print(f"\nГотово! Транскрибировано: {len(transcripts)} файлов → {OUTPUT_FILE}")

In [ ]:
# ── Ячейка 6: Проверка ───────────────────────────────────────────────────────
sample_key = list(transcripts.keys())[0]
sample_segs = transcripts[sample_key]

print(f"Пример: {sample_key}")
print(f"Сегментов: {len(sample_segs)}")
print()
for seg in sample_segs[:5]:
    print(f"  [{seg['start']:.1f}s → {seg['end']:.1f}s] {seg['text']}")